# 01 · Conversations and control

**Goal:** turn one-off requests into a real conversation, then shape how Claude responds and how the answer reaches your user.

| | |
|---|---|
| **You will learn** | Why the API has no memory, how to keep history, system prompts, what replaced `temperature`, streaming, and stop sequences |
| **Prerequisites** | [Notebook 00](00_setup_and_first_request.ipynb) |
| **Run time** | About 2 minutes |
| **Cost** | About $0.02 when I ran it. This notebook stops itself at $0.25 |
| **Paid cells** | Most cells in sections 1 to 7 make a small paid call |

If you haven't yet, set a spend limit in the [Claude Console](https://platform.claude.com/settings/billing) (**Billing**, then **Spend limits**).

## Setup

The same setup as notebook 00: the client, the budget, and two small functions. `track()` prints what a response cost and stops the notebook at `BUDGET_USD`. `text_from_message()` finds the answer among the response's blocks.

Run the cells from top to bottom: later sections reuse the functions and prompts defined earlier.

In [1]:
import os
import time

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)  # reads your keys from the .env file

api_key = os.getenv("ANTHROPIC_API_KEY", "")
assert api_key and api_key != "your_key_here", "Add your API key to .env, restart the kernel, and run this again."

client = Anthropic()  # finds ANTHROPIC_API_KEY by itself
MODEL = "claude-sonnet-5-5"

print("Ready. Using model:", MODEL)

Ready. Using model: claude-sonnet-5-5


In [2]:
INPUT_PRICE = 2.00    # dollars per 1 million input tokens (Sonnet 5.5)
OUTPUT_PRICE = 10.00  # dollars per 1 million output tokens (Sonnet 5.5)
BUDGET_USD = 0.25     # the most this notebook should spend

spent = 0.0


def track(response):
    global spent
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000
    spent += cost
    print(f"[input={input_tokens:,} output={output_tokens:,} cost=${cost:.5f} | spent ${spent:.4f} of ${BUDGET_USD:.2f}]")
    if spent > BUDGET_USD:
        raise RuntimeError(f"Budget reached: spent ${spent:.4f}. Raise BUDGET_USD if you mean to spend more.")
    return response


def text_from_message(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""

## 1. The API has no memory (paid)

**Key fact:** the API stores nothing between requests. Every request stands alone.

Imagine phoning someone who forgets everything the moment you hang up. To continue yesterday's chat, you first have to read them the whole transcript. That is exactly how the Claude API works.

Let's prove it. We tell Claude a fact, then ask about it in a second, separate request.

In [3]:
first = track(client.messages.create(
    model=MODEL,
    max_tokens=300,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "My favorite color is teal. Please remember that."}],
))
print("Claude:", text_from_message(first), "\n")

second = track(client.messages.create(
    model=MODEL,
    max_tokens=300,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "What is my favorite color?"}],
))
print("Claude:", text_from_message(second))

[input=23 output=143 cost=$0.00148 | spent $0.0015 of $0.25]
Claude: Teal is a great choice! It's a nice blend of blue and green, calm but still distinctive.

I should be upfront about one thing: I can remember it for the rest of this conversation, but I don't have memory that carries over between separate chats unless the app you're using has a memory feature enabled. If you start a new conversation later, you may need to remind me.

Is there something you'd like to use it for, like color palette ideas, decor, or design suggestions? 

[input=16 output=122 cost=$0.00125 | spent $0.0027 of $0.25]
Claude: I don't know your favorite color. I don't have any information about you beyond what you share in this conversation, and you haven't mentioned it.

If you'd like to tell me, I'm happy to chat about it. Or if you're trying to figure out a favorite, I can help with that, for example by talking through which colors you're drawn to in your clothes, home, or art.

What's your favorite color?

**What to notice**

- Claude may sound like it will remember in the first answer, but the second request knows nothing about it. It only received that one question.
- Apps like the Claude app feel like they remember because they resend the conversation for you. Next, we do the same thing ourselves.

## 2. Keeping the history yourself (paid)

To hold a conversation:

1. Keep a list of every message.
2. After each response, add Claude's reply to the list.
3. Add the next question, and send the **whole list** again.

Three small functions make this tidy. We use them in the notebooks that follow.

> **Different from the course:** the course stores only the answer's text. A response can also hold thinking blocks, and Anthropic's guidance is to send those back unchanged, not strip them out. So `add_assistant_message` stores the whole `response.content`.

In [5]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, response):
    messages.append({"role": "assistant", "content": response.content})


def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": MODEL,
        "max_tokens": 1000,
        "output_config": {"effort": "low"},
        "messages": messages,
    }
    # The API doesn't accept system=None, so only add these when they are set.
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    return track(client.messages.create(**params))

In [6]:
messages = []

add_user_message(messages, "My favorite color is teal. Please remember that.")
response = chat(messages)
add_assistant_message(messages, response)
print("Claude:", text_from_message(response), "\n")
first_input_tokens = response.usage.input_tokens

add_user_message(messages, "What is my favorite color?")
response = chat(messages)
add_assistant_message(messages, response)
answer = text_from_message(response)
print("Claude:", answer, "\n")

print("The history we just sent:")
for message in messages:
    content = message["content"]
    kinds = "text" if isinstance(content, str) else [block.type for block in content]
    print(f"  {message['role']:<10}{kinds}")

assert "teal" in answer.lower()
assert response.usage.input_tokens > first_input_tokens

[input=23 output=147 cost=$0.00152 | spent $0.0042 of $0.25]
Claude: Teal is a great choice. It's a nice blend of blue's calm and green's freshness.

I should be upfront about one thing: I can't carry memory between separate conversations, so I'll remember it for as long as we're chatting here, but a new conversation would start fresh. If you're using an app with a memory feature or custom instructions, you could save it there so it persists.

Is there something you'd like to do with it, like color palette ideas, decorating, or anything else? 

[input=180 output=12 cost=$0.00048 | spent $0.0047 of $0.25]
Claude: Your favorite color is teal. 

The history we just sent:
  user      text
  assistant ['text']
  user      text
  assistant ['text']


**What to notice**

- Same question as section 1, but now Claude knows the answer, because the first exchange was sent along with it.
- **The history shows `text` only?** That's normal at low effort, where Claude often skips thinking on easy questions. When a response does include thinking blocks, `add_assistant_message` keeps them too.
- **Look at `input` on the two cost lines.** It grows with every turn, because you pay to resend the whole history each time. Long conversations get more expensive per turn.

## 3. System prompts (paid)

A system prompt tells Claude **how to behave** for the whole conversation: its role, its tone, its rules. It goes in the separate `system` argument, not in `messages`.

Say you're building a math tutor. A good tutor gives hints and doesn't just hand over the answer. Compare the same question with and without a system prompt.

In [7]:
TUTOR_SYSTEM = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
Keep each reply under 60 words.
"""

question = "How do I solve 5x + 2 = 17 for x?"

print("--- Without a system prompt ---")
messages = []
add_user_message(messages, question)
print(text_from_message(chat(messages)))

print("\n--- With the tutor system prompt ---")
messages = []
add_user_message(messages, question)
print(text_from_message(chat(messages, system=TUTOR_SYSTEM)))

--- Without a system prompt ---
[input=27 output=152 cost=$0.00157 | spent $0.0063 of $0.25]
To solve 5x + 2 = 17, isolate x:

1. **Subtract 2 from both sides:**
 5x + 2 − 2 = 17 − 2
 5x = 15

2. **Divide both sides by 5:**
 5x ÷ 5 = 15 ÷ 5
 x = 3

**Check:** Plug it back in: 5(3) + 2 = 15 + 2 = 17 ✓

So **x = 3**.

--- With the tutor system prompt ---
[input=78 output=76 cost=$0.00092 | spent $0.0072 of $0.25]
Good question! The goal is to get x alone on one side.

Right now, 2 is being added to 5x. What operation could you do to *both sides* of the equation to cancel out that +2?

Try it, and tell me what new equation you get.


**What to notice**

- Without a system prompt, you likely get the full solution. With it, Claude asks you to take the next step.
- The user's question didn't change. Only the instructions around it did.
- **Keep the system prompt the same for a whole conversation.** Changing it midway invalidates earlier thinking blocks, and on newer accounts that request can fail with a 400.

## 4. Exercise: a chat loop (paid)

Everything so far, in one loop. This is the core of every chatbot.

By default the loop plays a short scripted conversation, so **Run All never gets stuck waiting for you**. To type your own messages, set `INTERACTIVE = True` and type `quit` to stop. Every turn costs a little and counts toward `BUDGET_USD`.

In [8]:
INTERACTIVE = False

scripted_turns = iter(["How do I solve 5x + 2 = 17 for x?", "Should I subtract 2 from both sides?", "Is x equal to 3?"])

messages = []
while True:
    user_input = input("You: ") if INTERACTIVE else next(scripted_turns, "quit")
    if user_input.strip().lower() == "quit":
        break
    if not INTERACTIVE:
        print("You:", user_input)

    add_user_message(messages, user_input)
    response = chat(messages, system=TUTOR_SYSTEM)
    add_assistant_message(messages, response)
    print("Claude:", text_from_message(response), "\n")

print(f"Conversation ended with {len(messages)} messages in the history.")

You: How do I solve 5x + 2 = 17 for x?
[input=78 output=97 cost=$0.00113 | spent $0.0083 of $0.25]
Claude: Good question! The goal is to get x alone on one side.

Right now, x is being multiplied by 5, and then 2 is added. To undo these, we work backward.

**Step 1:** What could you do to both sides of the equation to get rid of the "+ 2"? Try it and tell me what you get. 

You: Should I subtract 2 from both sides?
[input=191 output=99 cost=$0.00137 | spent $0.0097 of $0.25]
Claude: Yes, exactly! Subtracting 2 from both sides keeps the equation balanced.

**Step 2:** Do that now. What does the equation look like after subtracting 2 from each side?

Once you have it, think about what operation undoes "multiplied by 5." Tell me what you'd do next! 

You: Is x equal to 3?
[input=301 output=69 cost=$0.00129 | spent $0.0110 of $0.25]
Claude: Yes, x = 3 is correct. Nice work!

Let's verify it, which is a great habit. Substitute 3 for x in the original equation: what is 5(3) + 2? Does it matc

## 5. What happened to `temperature`? (paid)

The course teaches `temperature`, a dial from 0 to 1 for how predictable or creative answers are. **It no longer exists on the current API**, as you saw in notebook 00. The idea behind it is still useful, though.

For each next token, the model works out a probability for every option, then samples one:

```
"What do you think ___"     about 30%   would 20%   of 10%   ...
```

`temperature` used to change how strongly the model favored the top option. Now you steer through the **prompt**:

| You want | Old way | Today |
|---|---|---|
| Consistent, factual output | `temperature=0.0` | Clear instructions, and a fixed output format (notebook 02) |
| Creative variety | `temperature=1.0` | Ask for it: "unusual", "surprising", "give 5 different options" |

Sampling still happens, so the same prompt can give different answers. Run it twice, then steer it.

In [9]:
prompts = [
    "Give me a movie idea in one sentence.",
    "Give me a movie idea in one sentence.",
    "Give me a strange, surprising movie idea that nobody has made, in one sentence.",
]

for prompt in prompts:
    messages = []
    add_user_message(messages, prompt)
    print(f"Prompt: {prompt}")
    print(f"Claude: {text_from_message(chat(messages))}\n")

Prompt: Give me a movie idea in one sentence.
[input=22 output=61 cost=$0.00065 | spent $0.0117 of $0.25]
Claude: When a retired sound engineer discovers that the static on her late husband's old voicemail tapes is actually a coded message, she races across Europe to decode it before the people who erased his career find her first.

Prompt: Give me a movie idea in one sentence.
[input=22 output=75 cost=$0.00079 | spent $0.0125 of $0.25]
Claude: When a retired lighthouse keeper discovers that the beam he's tended for forty years has been quietly guiding lost souls home, he must decide whether to keep the light burning when a storm brings one final ship carrying someone from his past.

Prompt: Give me a strange, surprising movie idea that nobody has made, in one sentence.
[input=35 output=97 cost=$0.00104 | spent $0.0135 of $0.25]
Claude: A retired professional apology-writer for corporations discovers that his own estranged family has been secretly hiring *him*, through shell companies,

**What to notice**

- The first two prompts are identical, and the answers may or may not match. **You can't force identical output any more**, so test the things that matter with checks, not by comparing exact text. Notebook 03 is about exactly that.
- The third prompt shows that words in your prompt now do the job of the old dial.

## 6. Streaming (paid)

A long answer can take many seconds. Without streaming, your user stares at a spinner until the whole thing arrives. With streaming, words appear as they're generated, which feels far faster even when the total time is the same.

Run the next two cells and feel the difference. Both ask for the same story.

### 6a. Without streaming

In [ ]:
story_request = "Write a vivid 6-sentence story about a lighthouse keeper."

messages = []
add_user_message(messages, story_request)

print("Waiting for the whole response...\n")
start = time.perf_counter()
response = chat(messages)
elapsed = time.perf_counter() - start

print(text_from_message(response))
print(f"\nYou saw nothing for {elapsed:.1f}s, then it all appeared at once.")

### 6b. With streaming

`client.messages.stream(...)` gives you `text_stream`, which yields each piece of text as it arrives.

You still usually need the complete message afterwards, to save it or continue the conversation. `get_final_message()` gives you that, so you get the same object as a normal call. We pass it to `track()` for the cost, and to `add_assistant_message()` for the history.

In [11]:
story_request = "Write a vivid 6-sentence story about a lighthouse keeper."  # same request as 6a

messages = []
add_user_message(messages, story_request)

start = time.perf_counter()
first_text_at = None

with client.messages.stream(
    model=MODEL,
    max_tokens=1000,
    output_config={"effort": "low"},
    messages=messages,
) as stream:
    for text in stream.text_stream:
        if first_text_at is None:
            first_text_at = time.perf_counter() - start
        print(text, end="", flush=True)
    final_message = stream.get_final_message()

total = time.perf_counter() - start
print(f"\n\nFirst words after {first_text_at:.1f}s. Finished at {total:.1f}s.")
track(final_message)

# The final message continues a conversation like any other response.
add_assistant_message(messages, final_message)
add_user_message(messages, "Give that story a title. Reply with the title only.")
print("\nTitle:", text_from_message(chat(messages)))

assert first_text_at < total

Each night, Elias climbed the 212 iron steps of Greyfell Light, his knees complaining louder than the wind that howled through the cracks in the stone. Tonight the storm had swallowed the horizon whole, and the sea heaved against the rocks like a living thing trying to break its chains. He lit the great lamp, and its beam sliced through the dark, sweeping over waves that glittered like broken glass. Far out, a tiny orange flare bloomed and vanished, a fishing boat's desperate whisper in the roar. Elias gripped the rail, rotating the lens by hand when the gears seized, his palms blistering as he refused to let the light falter for even a heartbeat. By dawn, the storm had worn itself out, and below him a battered little trawler limped into the harbor, its captain raising a weary hand toward the tower that had held steady all night.

First words after 1.0s. Finished at 4.2s.
[input=32 output=282 cost=$0.00288 | spent $0.0164 of $0.25]
[input=331 output=8 cost=$0.00074 | spent $0.0171 of $

**What to notice**

- Compare "first words" with "finished". The gap between them is time your user spent reading, not waiting.
- Thinking happens before the text. At higher effort, there can be a longer pause before the first words appear.

### 6c. Under the hood

`text_stream` is a convenience. A stream is really a series of **events**, and you can see them by passing `stream=True` to `create()`:

| Event | Meaning |
|---|---|
| `message_start` | A message has begun. It carries the input token count |
| `content_block_start` | A new block has begun, such as text or thinking |
| `content_block_delta` | A small piece of that block. This is where the text is |
| `content_block_stop` | That block is finished |
| `message_delta` | Final details: the `stop_reason` and the output token count |
| `message_stop` | The message is finished |

This low-level form doesn't build a final message for you, so to track the cost we keep the message from `message_start` and fill in the final output count ourselves.

In [12]:
stream = client.messages.create(
    model=MODEL,
    max_tokens=200,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "Say hello in one short sentence."}],
    stream=True,
)

message = None
text = ""
for event in stream:
    print(event.type, end="")
    if event.type == "message_start":
        message = event.message
    elif event.type == "content_block_delta":
        print(f"  ({event.delta.type})", end="")
        if event.delta.type == "text_delta":
            text += event.delta.text
    elif event.type == "message_delta":
        print(f"  (stop_reason={event.delta.stop_reason})", end="")
        message.usage.output_tokens = event.usage.output_tokens
    print()

print("\nText we collected:", text)
track(message)

assert text

message_start
content_block_start
content_block_delta  (text_delta)
content_block_delta  (text_delta)
content_block_delta  (text_delta)
content_block_stop
message_delta  (stop_reason=end_turn)
message_stop

Text we collected: Hello, it's nice to meet you!
[input=19 output=15 cost=$0.00019 | spent $0.0173 of $0.25]


## 7. Stop sequences (paid)

`stop_sequences` tells Claude to stop as soon as it writes a given string. It's useful for cutting off output at a known marker.

In [13]:
messages = []
add_user_message(messages, "Count from 1 to 10, one number per line. Output only the numbers.")
response = chat(messages, stop_sequences=["5"])
text = text_from_message(response)

print("What came back:")
print(text)
print("\nstop_reason  :", response.stop_reason)
print("stop_sequence:", response.stop_sequence)

assert response.stop_reason == "stop_sequence"
assert response.stop_sequence == "5"
assert "5" not in text

[input=31 output=21 cost=$0.00027 | spent $0.0176 of $0.25]
What came back:
1
2
3
4


stop_reason  : stop_sequence
stop_sequence: 5


**What to notice**

- Output stopped just before the `5`. **The stop string itself is not included** in the text.
- `stop_reason` tells you it stopped on a sequence, and `stop_sequence` tells you which one, if you passed several.

> The course pairs stop sequences with **assistant prefill** to get clean JSON. Prefill now returns a 400 on current models, including Sonnet 5.5. Notebook 02 shows what replaced it.

## Final spend

In [14]:
print(f"This notebook spent ${spent:.4f} of its ${BUDGET_USD:.2f} budget.")

This notebook spent $0.0176 of its $0.25 budget.


## Takeaways

1. **The API has no memory.** You keep the history and resend all of it, so each turn costs a bit more than the last.
2. **Store the whole `response.content`**, not just the text, so thinking blocks go back unchanged.
3. **System prompts set the behavior.** They go in `system`, and they should stay the same for the whole conversation.
4. **`temperature` is gone.** Steer with your prompt, and don't expect identical output between runs.
5. **Streaming improves how fast an answer feels.** Use `get_final_message()` when you also need the complete response.
6. **Stop sequences cut output at a marker**, and the marker isn't included.

## Your turn

Each exercise costs about a cent or less.

- In section 3, write a system prompt for a pirate who only answers in rhymes. Then ask the same math question.
- In section 4, set `INTERACTIVE = True` and have your own chat with the tutor. Watch `input` grow on each turn.
- In section 5, change the third prompt to ask for five different ideas. How does the output cost change?
- In section 6b, change `"low"` to `"high"`. How much longer do you wait for the first words?
- In section 7, change the stop sequence to `["\n"]`. What comes back?

**Next:** [02 · Structured outputs](02_structured_outputs.ipynb): getting validated JSON from Claude with Pydantic.